# LOT 2026 - Cognitive modeling meets computational linguistics (Yevgen Matusevych)
## Day 3 practical
# Probing phonetic structure in a self-supervised speech model

This notebook reads out the internal representations of a self-supervised speech model
(wav2vec 2.0, trained only to predict masked speech) and measures, layer by layer, how well each
layer discriminates vowels. The discrimination measure is the ABX task, which operates on a
continuous representation space and does not assume that the model has formed discrete categories.

The expected outcome is a non-monotonic profile: vowel discriminability increases over the early
and middle layers and decreases toward the output. Phonetic contrasts are therefore most
recoverable from the model's intermediate representations, even though the model was never trained
on phonetic labels or categories.

Set the runtime to GPU: Runtime > Change runtime type > GPU. CPU also works but is slower.

The default dataset is the Hillenbrand et al. (1995) vowels. If the download fails, Section 3
   lists manual options, or you can switch to a backup dataset by changing one line.

Each step operates on a list of (audio, label, speaker) items, so changing the dataset does not
affect the rest of the notebook.

## 0. Setup

In [ ]:
# Colab includes torch and torchaudio. Install the remaining packages.
!pip install -q "transformers>=4.40" "datasets>=2.18" "huggingface_hub>=0.23" librosa soundfile scikit-learn tqdm

import io
import os
import glob
import re
import random
import urllib.request
import zipfile
import warnings
from collections import defaultdict

import librosa
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from tqdm.auto import tqdm
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor

warnings.filterwarnings("ignore")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} | device: {device}")

## 1. Configuration

All adjustable settings are collected here. The defaults run in a few minutes on a GPU.

In [ ]:
# Model
MODEL_NAME = "facebook/wav2vec2-base"      # alternative: "facebook/hubert-base-ls960"

# Dataset
DATA_SOURCE = "hillenbrand"                # "hillenbrand" or "speech_commands" (backup)
HILLENBRAND_DIR = "/content/hillenbrand"   # download / upload location
HILLENBRAND_GROUPS = ["m", "w"]            # m=men, w=women, b=boys, g=girls
MAX_TOKENS_PER_LABEL = None                # cap tokens per vowel for speed; None uses all

# Speech Commands backup (streamed)
SPEECH_COMMANDS_WORDS = ["yes", "no", "up", "down", "left", "right", "go", "stop"]
SPEECH_COMMANDS_PER_WORD = 30

# ABX
N_TRIPLETS = 3000                          # triplets sampled per layer
ACROSS_SPEAKER = True                      # require A and X from different speakers

# Visualisation
PCA_LAYER = None                           # layer to inspect; None uses the ABX peak from Section 6

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 2. Hugging Face token (optional)

The default models are public and do not require a token. Provide one only if you switch to a
gated model or encounter download rate limits. The recommended approach is to store the token in
Colab Secrets (the key icon in the left sidebar) under the name HF_TOKEN. Alternatively, paste it
into the variable below.

In [ ]:
HF_TOKEN = ""  # optional; prefer a Colab Secret named "HF_TOKEN"

def resolve_hf_token():
    """Return a Hugging Face token from the variable or a Colab Secret, or None."""
    if HF_TOKEN:
        return HF_TOKEN
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        return None

token = resolve_hf_token()
if token:
    from huggingface_hub import login
    login(token)
    print("Logged in to Hugging Face.")
else:
    print("No token found; the default public models do not require one.")

## 3. Load the data

Each loader returns a list of items. An item is a dictionary with a label, a speaker, and either a
file path or an in-memory array with its sampling rate.

The Hillenbrand audio is downloaded from a GitHub mirror that is hosted with the author's
permission (santiagobarreda/hillenbrand_et_al_1995, MIT-licensed). The bundle contains nested
archives for men, women, and children, which are unpacked automatically. Filenames encode the
speaker and vowel: for example, m01ae.wav is man 01 producing the vowel in "had". Note that the
audio files use the code "ei" for the vowel in "hayed", although the original readme text lists it
as "ey".

In [ ]:
# The twelve Hillenbrand vowels, with readable names for the plots.
VOWEL_NAMES = {
    "iy": "i (heed)",   "ih": "\u026a (hid)",  "eh": "\u025b (head)",  "ae": "\u00e6 (had)",
    "ah": "\u0251 (hod)", "aw": "\u0254 (hawed)", "uh": "\u028c (hud)",  "oo": "\u028a (hood)",
    "uw": "u (who'd)",  "er": "\u025d (heard)", "ei": "e\u026a (hayed)", "oa": "o\u028a (boat)",
}

HILLENBRAND_URLS = [
    "https://github.com/santiagobarreda/hillenbrand_et_al_1995/raw/main/h95-alldata.zip",
    "https://raw.githubusercontent.com/santiagobarreda/hillenbrand_et_al_1995/main/h95-alldata.zip",
]
FILENAME_PATTERN = re.compile(r"^([mwbg])(\d{2})([a-z]{2})\.wav$", re.IGNORECASE)

MANUAL_INSTRUCTIONS = (
    "Could not download the Hillenbrand audio automatically. Options:\n"
    "  (A) Download 'h95-alldata.zip' from\n"
    "      https://github.com/santiagobarreda/hillenbrand_et_al_1995 (author-permitted mirror),\n"
    "      upload it into HILLENBRAND_DIR via the Colab Files panel, and re-run this cell.\n"
    "  (B) Or set DATA_SOURCE = 'speech_commands' in the configuration and re-run."
)


def find_wav_files(folder):
    """Return every .wav path under `folder`."""
    return [os.path.join(root, name)
            for root, _, files in os.walk(folder)
            for name in files if name.lower().endswith(".wav")]


def unzip_everything(folder):
    """Recursively unzip every .zip file under `folder`."""
    while True:
        archives = [os.path.join(root, name)
                    for root, _, files in os.walk(folder)
                    for name in files if name.lower().endswith(".zip")]
        if not archives:
            return
        for path in archives:
            with zipfile.ZipFile(path) as archive:
                archive.extractall(os.path.dirname(path))
            os.remove(path)


def download_hillenbrand(folder):
    """Download and unpack the Hillenbrand audio bundle into `folder`."""
    os.makedirs(folder, exist_ok=True)
    for url in HILLENBRAND_URLS:
        try:
            print("Downloading", url)
            request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            data = urllib.request.urlopen(request, timeout=60).read()
            zipfile.ZipFile(io.BytesIO(data)).extractall(folder)
            unzip_everything(folder)
            return
        except Exception as error:
            print("  failed:", repr(error)[:140])


def load_hillenbrand():
    """Load Hillenbrand vowels as (path, vowel, speaker) items."""
    if os.path.isdir(HILLENBRAND_DIR):
        unzip_everything(HILLENBRAND_DIR)
    wav_files = find_wav_files(HILLENBRAND_DIR)
    if not wav_files:
        download_hillenbrand(HILLENBRAND_DIR)
        wav_files = find_wav_files(HILLENBRAND_DIR)
    if not wav_files:
        raise RuntimeError(MANUAL_INSTRUCTIONS)

    items = []
    for path in wav_files:
        match = FILENAME_PATTERN.match(os.path.basename(path))
        if match is None:
            continue
        group, talker, vowel = match.group(1).lower(), match.group(2), match.group(3).lower()
        if group in HILLENBRAND_GROUPS and vowel in VOWEL_NAMES:
            items.append({"path": path, "label": vowel, "speaker": group + talker})
    return items


def load_speech_commands():
    """Load a few isolated spoken words from Google Speech Commands (streamed)."""
    from datasets import load_dataset
    dataset = load_dataset("speech_commands", "v0.02", split="train", streaming=True)
    label_names = dataset.features["label"].names

    remaining = {word: SPEECH_COMMANDS_PER_WORD for word in SPEECH_COMMANDS_WORDS}
    items = []
    for example in dataset:
        word = label_names[example["label"]]
        if remaining.get(word, 0) > 0:
            audio = example["audio"]
            items.append({"array": np.asarray(audio["array"], dtype=np.float32),
                          "sr": audio["sampling_rate"],
                          "label": word,
                          "speaker": str(example.get("speaker_id", "unknown"))})
            remaining[word] -= 1
        if all(count == 0 for count in remaining.values()):
            break
    return items


def cap_per_label(items, max_per_label):
    """Keep at most `max_per_label` randomly chosen items per label."""
    shuffled = random.Random(SEED).sample(items, len(items))
    kept = defaultdict(int)
    capped = []
    for item in shuffled:
        if kept[item["label"]] < max_per_label:
            capped.append(item)
            kept[item["label"]] += 1
    return capped

In [ ]:
loaders = {"hillenbrand": load_hillenbrand, "speech_commands": load_speech_commands}
if DATA_SOURCE not in loaders:
    raise ValueError("DATA_SOURCE must be 'hillenbrand' or 'speech_commands'.")

items = loaders[DATA_SOURCE]()
if MAX_TOKENS_PER_LABEL:
    items = cap_per_label(items, MAX_TOKENS_PER_LABEL)

labels = np.array([item["label"] for item in items])
speakers = np.array([item["speaker"] for item in items])

print(f"{len(items)} tokens | {len(set(labels))} labels | {len(set(speakers))} speakers")
print("labels:", sorted(set(labels)))

## 4. Load the model

Setting `output_hidden_states=True` returns the output of every layer from a single forward pass.
A base model produces 13 representations: the input projection (layer 0) and 12 transformer
layers.

In [ ]:
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_NAME)
model = Wav2Vec2Model.from_pretrained(MODEL_NAME, output_hidden_states=True).eval().to(device)

n_layers = model.config.num_hidden_layers + 1
hidden_size = model.config.hidden_size
print(f"{MODEL_NAME}: {n_layers} layers (0..{n_layers - 1}), hidden size {hidden_size}")

## 5. Extract one vector per token, per layer

Each token is passed through the model once, and the output at every layer is averaged over time
(mean pooling) to give a single vector per layer.

Mean pooling discards temporal dynamics. The ABX evaluation in Schatz et al. (2021) compares full
frame sequences using dynamic time warping. Mean pooling is faster and adequate for isolated
vowels; a frame-level version is listed in the extensions.

In [ ]:
def load_waveform(item, sample_rate=16000):
    """Return the item's audio as a 16 kHz mono waveform."""
    if "path" in item:
        signal, _ = librosa.load(item["path"], sr=sample_rate, mono=True)
        return signal
    signal = item["array"]
    if item["sr"] != sample_rate:
        signal = librosa.resample(signal, orig_sr=item["sr"], target_sr=sample_rate)
    return signal


@torch.no_grad()
def layer_embeddings(signal):
    """Return mean-pooled features for every layer: shape (n_layers, hidden_size)."""
    inputs = feature_extractor(signal, sampling_rate=16000, return_tensors="pt")
    hidden_states = model(inputs.input_values.to(device)).hidden_states
    return np.stack([layer.mean(dim=1).squeeze(0).cpu().numpy() for layer in hidden_states])


# features[layer] is a (n_tokens, hidden_size) matrix.
features = np.zeros((n_layers, len(items), hidden_size), dtype=np.float32)
for i, item in enumerate(tqdm(items, desc="Extracting")):
    features[:, i, :] = layer_embeddings(load_waveform(item))

print("features:", features.shape, "(layers, tokens, dim)")

## 6. ABX discriminability, layer by layer

The ABX task uses three tokens. A and X share a label; B has a different label. The representation
is scored as correct when X is closer to A than to B. Averaging over many triplets gives a score
per layer, where 0.5 is chance and 1.0 is perfect discrimination.

When ACROSS_SPEAKER is True, A and X are drawn from different speakers, so the score reflects
phonetic structure rather than speaker identity.

In [ ]:

def cosine_distance(a, b):
    """Cosine distance between two vectors (0 = same direction, 2 = opposite)."""
    a = a / (np.linalg.norm(a) + 1e-9)
    b = b / (np.linalg.norm(b) + 1e-9)
    return 1.0 - float(a @ b)


def abx_accuracy(layer_features, labels, speakers, n_triplets, across_speaker, seed=0):
    """Fraction of ABX triplets the representation gets right (chance = 0.5).

    A and X share a label, B differs; correct when X is nearer to A than to B.
    When across_speaker is True, A and X are drawn from different speakers.
    """
    rng = np.random.default_rng(seed)

    tokens_by_label = defaultdict(list)
    speakers_by_label = defaultdict(lambda: defaultdict(list))
    for index, (label, speaker) in enumerate(zip(labels, speakers)):
        tokens_by_label[label].append(index)
        speakers_by_label[label][speaker].append(index)

    if across_speaker:
        usable = [label for label in tokens_by_label if len(speakers_by_label[label]) >= 2]
    else:
        usable = [label for label in tokens_by_label if len(tokens_by_label[label]) >= 2]

    correct = 0
    for _ in range(n_triplets):
        same = rng.choice(usable)
        other = rng.choice([label for label in usable if label != same])

        if across_speaker:
            speaker_a, speaker_x = rng.choice(list(speakers_by_label[same]), size=2, replace=False)
            a = rng.choice(speakers_by_label[same][speaker_a])
            x = rng.choice(speakers_by_label[same][speaker_x])
        else:
            a, x = rng.choice(tokens_by_label[same], size=2, replace=False)
        b = rng.choice(tokens_by_label[other])

        distance_to_a = cosine_distance(layer_features[a], layer_features[x])
        distance_to_b = cosine_distance(layer_features[b], layer_features[x])
        if distance_to_a < distance_to_b:
            correct += 1
    return correct / n_triplets


abx_scores = [abx_accuracy(features[layer], labels, speakers, N_TRIPLETS, ACROSS_SPEAKER, SEED)
              for layer in range(n_layers)]

for layer, score in enumerate(abx_scores):
    print(f"layer {layer:2d}: ABX = {score:.3f}")

best_layer = int(np.argmax(abx_scores))
print(f"\nBest layer: {best_layer} (ABX = {abx_scores[best_layer]:.3f})")

## 7. Discriminability across layers

The plot shows ABX accuracy as a function of layer depth. The expected shape is non-monotonic:
discriminability rises into the middle layers and decreases toward the output. The model was
trained without phonetic labels or categories, yet phonetic contrasts are most recoverable from
its intermediate layers.

In [ ]:
plt.figure(figsize=(7, 4.2))
plt.plot(range(n_layers), abx_scores, "o-", linewidth=2)
plt.axhline(0.5, linestyle="--", color="gray", linewidth=1, label="chance")
plt.scatter([best_layer], [abx_scores[best_layer]], s=140, facecolors="none",
            edgecolors="crimson", linewidth=2, zorder=5, label=f"peak (layer {best_layer})")
plt.xlabel("layer  (0 = input projection)")
plt.ylabel("ABX accuracy")
plt.title(f"Phonetic discriminability across layers\n{MODEL_NAME}")
plt.ylim(0.45, 1.02)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 8. The representation space (PCA)

This is a two-dimensional PCA projection of one layer, with points coloured by label. It gives a
qualitative view of how the vowels are arranged: whether they form separated clusters or a
continuous space with denser regions. The ABX measure above does not depend on this distinction,
since it is computed directly from distances.

In [ ]:
plot_layer = best_layer if PCA_LAYER is None else PCA_LAYER
unique_labels = sorted(set(labels))
palette = plt.cm.tab20(np.linspace(0, 1, len(unique_labels)))
coordinates = PCA(n_components=2, random_state=SEED).fit_transform(features[plot_layer])

plt.figure(figsize=(7.5, 6))
for color, label in zip(palette, unique_labels):
    points = coordinates[labels == label]
    plt.scatter(points[:, 0], points[:, 1], s=18, alpha=0.7, color=color,
                label=VOWEL_NAMES.get(label, label))
plt.title(f"PCA of layer {plot_layer}  ({MODEL_NAME})")
plt.xlabel("PC 1")
plt.ylabel("PC 2")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

## 9. Optional: distances between categories

This computes the cosine distance between the centroid of each label at the chosen layer. Smaller
distances (darker cells) indicate categories that lie closer together and are therefore more
easily confused. For vowels, this gives a view of the representation as a similarity space, in
which confusability is determined by geometry rather than by explicit category boundaries.

In [ ]:
plot_layer = best_layer if PCA_LAYER is None else PCA_LAYER
unique_labels = sorted(set(labels))
centroids = np.stack([features[plot_layer][labels == label].mean(axis=0) for label in unique_labels])
distances = np.array([[cosine_distance(p, q) for q in centroids] for p in centroids])
np.fill_diagonal(distances, np.nan)
names = [VOWEL_NAMES.get(label, label) for label in unique_labels]

plt.figure(figsize=(7.5, 6.2))
image = plt.imshow(distances, cmap="viridis")
plt.colorbar(image, label="cosine distance between centroids")
plt.xticks(range(len(names)), names, rotation=90, fontsize=8)
plt.yticks(range(len(names)), names, fontsize=8)
plt.title(f"Pairwise label distance at layer {plot_layer}")
plt.tight_layout()
plt.show()

## 10. Categorical perception: identification and discrimination

A discrimination measure on its own cannot separate a categorical representation from a continuous
one. The classic test does it by comparing two functions: an identification function (which category
each step belongs to) and a discrimination function (how well neighbouring steps are told apart).
Categorical perception is the finding that discrimination is predictable from identification. Identification curve should be sigmoid-shaped: easy at the category center and difficult at the boundary.
Discrimination should peak at the category boundary and be poor within a category, exactly as it would be
if a listener only had access to the category label.

This section builds both functions for the model on a synthetic /l/ - /r/ continuum, and compares the
obtained discrimination against the discrimination predicted from identification. A model that
behaves categorically shows the two tracking each other; a model with a continuous representation
discriminates within categories more than identification predicts.

The continuum stimuli are the /lih/ - /rih/ synthesized from de Heer Kloots and Zuidema (2024). As before, the whole word is given to the model and the representation is read out
over the liquid segment marked in each stimulus's TextGrid. Consonant contrasts like /l/ - /r/ are
perceived more categorically than vowels, so this is a good place to look for the effect.

Pleaes cite de Heer Kloots & Zuidema (2024) for the stimuli, and do not redistribute them without the authors' permission.

In [ ]:
CONTINUUM_URL = "https://github.com/mdhk/phonotactic-sensitivity/archive/refs/heads/main.zip"
CONTINUUM_DIR = "/content/phonotactic"
CONTINUUM_CONTEXT = "lih-rih"   # the neutral /l/-/r/ continuum (no biasing onset)


def download_continua(folder):
    """Download the /l/-/r/ continuum stimuli into `folder`."""
    if glob.glob(os.path.join(folder, "**", "cntnm_*.wav"), recursive=True):
        return
    os.makedirs(folder, exist_ok=True)
    request = urllib.request.Request(CONTINUUM_URL, headers={"User-Agent": "Mozilla/5.0"})
    data = urllib.request.urlopen(request, timeout=120).read()
    zipfile.ZipFile(io.BytesIO(data)).extractall(folder)


def load_continuum(folder, context):
    """Return one onset context's continuum stimuli, with step (0..10) and voice."""
    items = []
    pattern = os.path.join(folder, "**", "steps", f"cntnm_{context}_*.wav")
    for path in glob.glob(pattern, recursive=True):
        step = int(os.path.basename(path).rsplit("_", 1)[1].split(".")[0])
        voice = "A" if "voiceA" in path else "E"
        items.append({"path": path, "step": step, "voice": voice})
    return items


def textgrid_x_interval(path):
    """Return (start, end) in seconds of the interval labelled 'X' (the liquid)."""
    with open(path) as handle:
        text = handle.read()
    blocks = re.findall(r'xmin = ([\d.]+)\s*\n\s*xmax = ([\d.]+)\s*\n\s*text = "([^"]*)"', text)
    for start, end, label in blocks:
        if label == "X":
            return float(start), float(end)
    return None


def liquid_window(item, context):
    """Interpolate the liquid window for a step from the two endpoint TextGrids."""
    voice_dir = os.path.dirname(os.path.dirname(item["path"]))
    start0, end0 = textgrid_x_interval(os.path.join(voice_dir, f"{context}_cntnm_000.TextGrid"))
    start10, end10 = textgrid_x_interval(os.path.join(voice_dir, f"{context}_cntnm_010.TextGrid"))
    fraction = item["step"] / 10.0
    return (start0 + fraction * (start10 - start0),
            end0 + fraction * (end10 - end0))


@torch.no_grad()
def liquid_embeddings(signal, start, end):
    """Mean-pool each layer over the frames inside [start, end] seconds (the liquid)."""
    inputs = feature_extractor(signal, sampling_rate=16000, return_tensors="pt")
    hidden_states = model(inputs.input_values.to(device)).hidden_states
    n_frames = hidden_states[0].shape[1]
    frame_times = (np.arange(n_frames) + 0.5) * (len(signal) / 16000) / n_frames
    mask = (frame_times >= start) & (frame_times <= end)
    if not mask.any():
        mask[:] = True
    return np.stack([h[0, mask].mean(dim=0).cpu().numpy() for h in hidden_states])


download_continua(CONTINUUM_DIR)
continuum = sorted(load_continuum(CONTINUUM_DIR, CONTINUUM_CONTEXT),
                   key=lambda item: (item["voice"], item["step"]))
n_voices = len(set(item["voice"] for item in continuum))
n_steps = len(set(item["step"] for item in continuum))
print(f"{len(continuum)} stimuli for context {CONTINUUM_CONTEXT} ({n_voices} voices x {n_steps} steps)")

In [ ]:
cont_features = np.zeros((n_layers, len(continuum), hidden_size), dtype=np.float32)
for i, item in enumerate(tqdm(continuum, desc="Continuum")):
    start, end = liquid_window(item, CONTINUUM_CONTEXT)
    cont_features[:, i, :] = liquid_embeddings(load_waveform(item), start, end)

cont_step = np.array([item["step"] for item in continuum])
cont_voice = np.array([item["voice"] for item in continuum])
print("cont_features:", cont_features.shape, "(layers, stimuli, dim)")

The identification function uses the continuum endpoints (step 0 and step 10, pooled over
voices) as the /l/ and /r/ references, giving a soft P(R) for each step. Discrimination is the
representational distance between steps two apart, averaged over voices. Predicted discrimination is
the change in identification across that same two-step span, which is what label-based discrimination
would give.

In [ ]:
CONTINUUM_LAYER = 7
STEP_GAP = 2   # compare stimuli this many steps apart (Liberman used a two-step comparison)


def identification(layer_features, steps, layer):
    """Soft P(R) per step: proximity to the /r/ endpoint relative to the /l/ endpoint."""
    f = layer_features[layer]
    l_anchor = f[steps == 0].mean(axis=0)
    r_anchor = f[steps == steps.max()].mean(axis=0)
    scores = []
    for step in range(steps.max() + 1):
        vector = f[steps == step].mean(axis=0)
        d_l = cosine_distance(vector, l_anchor)
        d_r = cosine_distance(vector, r_anchor)
        scores.append(d_l / (d_l + d_r))
    return np.array(scores)


def discrimination(layer_features, steps, voices, layer, gap):
    """Representational distance between stimuli `gap` steps apart, averaged over voices."""
    f = layer_features[layer]
    n = steps.max() + 1
    per_voice = []
    for voice in sorted(set(voices)):
        reps = [f[(voices == voice) & (steps == s)][0] for s in range(n)]
        per_voice.append([cosine_distance(reps[s], reps[s + gap]) for s in range(n - gap)])
    return np.array(per_voice).mean(axis=0)


def boundary_step(p_r):
    """Continuum position where identification crosses 0.5."""
    for step in range(len(p_r) - 1):
        a, b = p_r[step] - 0.5, p_r[step + 1] - 0.5
        if a == 0:
            return float(step)
        if a * b < 0:
            return step + a / (a - b)
    return None


p_r = identification(cont_features, cont_step, CONTINUUM_LAYER)
obtained = discrimination(cont_features, cont_step, cont_voice, CONTINUUM_LAYER, STEP_GAP)
predicted = np.abs(p_r[STEP_GAP:] - p_r[:-STEP_GAP])
positions = np.arange(len(obtained)) + STEP_GAP / 2.0
boundary = boundary_step(p_r)

fig, (top, bottom) = plt.subplots(2, 1, figsize=(7, 6.6), sharex=True)

top.plot(range(len(p_r)), p_r, "o-")
top.axhline(0.5, linestyle="--", color="gray", linewidth=1)
if boundary is not None:
    top.axvline(boundary, linestyle=":", color="black", linewidth=1)
top.set_ylabel("identification  P(R)")
top.set_ylim(-0.02, 1.02)
top.set_title(f"Categorical-perception test on the {CONTINUUM_CONTEXT} continuum\n"
              f"layer {CONTINUUM_LAYER}, {MODEL_NAME}")
top.grid(alpha=0.3)

bottom.plot(positions, obtained / obtained.max(), "o-", label="obtained (model)")
bottom.plot(positions, predicted / (predicted.max() + 1e-9), "s--", label="predicted from identification")
if boundary is not None:
    bottom.axvline(boundary, linestyle=":", color="black", linewidth=1, label="identification boundary")
bottom.set_xlabel("continuum position  (0 = clear l, 10 = clear r)")
bottom.set_ylabel(f"{STEP_GAP}-step discrimination (normalised)")
bottom.legend(fontsize=8)
bottom.grid(alpha=0.3)

fig.tight_layout()
plt.show()

The next cell measures, at each layer, how peaked the obtained discrimination is (its maximum
divided by its mean). A value near 1 is flat (continuous); a larger value means a boundary peak. Use
it to choose an informative layer, then read the two-panel plot above at that layer.

In [ ]:
def discrimination_peakedness(layer):
    d = discrimination(cont_features, cont_step, cont_voice, layer, STEP_GAP)
    return d.max() / d.mean()


profile = [discrimination_peakedness(layer) for layer in range(n_layers)]

plt.figure(figsize=(7, 4))
plt.plot(range(n_layers), profile, "o-")
plt.axhline(1.0, linestyle="--", color="gray", linewidth=1, label="flat (continuous)")
plt.xlabel("layer")
plt.ylabel("discrimination peakedness (max / mean)")
plt.title("Boundary peakedness across layers")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### How to read this

The top panel is the model's identification function: how r-like each step is, with the boundary
(the 0.5 crossing) marked. The bottom panel compares two discrimination curves. The obtained curve
is how much the model's representation actually changes across a two-step span. The predicted curve
is what discrimination should look like if the model relied only on the category of each stimulus; it
peaks at the boundary by construction.

If the obtained curve tracks the predicted one, peaking at the boundary and falling within each
category, the model discriminates as if it only had category labels: the categorical-perception
signature. If the obtained curve stays flatter than the predicted one, the model discriminates
within categories better than labels would allow, so within-category detail is preserved: the
continuous, perceptual-space pattern.

This obtained-versus-predicted comparison is the version of "discrimination tracks identification"
from the classic categorical-perception studies, applied to the model, and it is the most direct
test of the categorical-versus-continuous question in the readings. Both curves are normalised, so
read the shape: whether obtained follows predicted, or stays flat where predicted peaks.

## 11. Extensions and discussion

A useful question to consider is which analysis choices affected the result. Re-running with a
different PCA_LAYER, with ACROSS_SPEAKER set to False, or with a Euclidean distance can change the
picture. If it does, that has implications for any claim of the form "the model represents vowels".

Suggested extensions, each a small change to the code:

- Different model: set MODEL_NAME to "facebook/hubert-base-ls960" and compare the layer profile.
  The pre-training objective influences where phonetic information is most accessible.
- Native versus non-native contrasts: add a contrast that is phonemic in one language but not
  another and compare ABX scores. This relates to perceptual narrowing (Werker & Tees, 1984).
- Frame-level distances: replace mean pooling with full frame sequences and a dynamic time warping
  distance, which is closer to existing ABX evaluations.
- Variability: average abx_accuracy over several random seeds and plot a confidence band.

References: Pasad, Chou & Livescu (2021) for the layer-wise analysis; Schatz et al. (2021) and Matusevych et al. (2023) for ABX and perceptual-space learning; Hillenbrand et al. (1995) for the vowel data; de Heer Kloots and Zuidema (2024) for the lih-rih stimuli.